In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Author: Manou Liesker. Student number: 15250946

In [ ]:
NETWORK_FOLDER = Path(r"Z:\Video_Files")
scale = 0.5  # 50% size

cap = cv2.VideoCapture(str(NETWORK_FOLDER / "Test.avi"))
object_detector = cv2.createBackgroundSubtractorMOG2()


while True:
    ret, frame = cap.read()
    if not ret:
        break

    small = cv2.resize(frame, None,  fx=scale, fy=scale)
    mask = object_detector.apply(small)

    cv2.imshow("Frame", small)
    cv2.imshow("Mask", mask)

    key = cv2.waitKey(30)
    if key == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()


In [8]:
NETWORK_FOLDER = Path(r"Z:\Video_Files")
input_path = str(NETWORK_FOLDER / "Test.avi")
output_path = str(NETWORK_FOLDER / "Test_masked3.avi")

cap = cv2.VideoCapture(input_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"XVID")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height), isColor=True)

scale = 0.5  # 50% size

while True:

    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray_blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # dark object on bright background
    mask = object_detector.apply(gray_blur)

    # make a visible masked result for presentation
    masked_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)

    out.write(masked_bgr)

cap.release()
out.release()

print("Saved to:", output_path)

Saved to: Z:\Video_Files\Test_masked3.avi


In [9]:

# =========================
# SETTINGS
# =========================

NETWORK_FOLDER = Path(r"Z:\Video_Files")
input_path = str(NETWORK_FOLDER / "Test.avi")
output_path = str(NETWORK_FOLDER / "Test_ball_only.avi")

THRESHOLD_VALUE = 90       # adjust if needed
BLUR_KERNEL = 5            # must be odd
MIN_AREA = 100             # reject tiny blobs
MAX_AREA = 20000           # reject huge blobs
MIN_CIRCULARITY = 0.45     # 1 = perfect circle

# =========================
# HELPER FUNCTIONS
# =========================

def contour_circularity(cnt):
    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)
    if perimeter == 0:
        return 0
    return 4 * np.pi * area / (perimeter ** 2)

def choose_ball_contour(contours):
    candidates = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < MIN_AREA or area > MAX_AREA:
            continue

        circ = contour_circularity(cnt)
        if circ < MIN_CIRCULARITY:
            continue

        candidates.append((cnt, area, circ))

    if not candidates:
        return None

    # Prefer most circular contour, then larger area
    candidates.sort(key=lambda x: (-x[2], -x[1]))
    return candidates[0][0]

# =========================
# LOAD VIDEO
# =========================

cap = cv2.VideoCapture(input_path)
if not cap.isOpened():
    raise RuntimeError(f"Could not open input video: {input_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"XVID")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height), isColor=True)

if not out.isOpened():
    cap.release()
    raise RuntimeError(f"Could not open output video: {output_path}")

# =========================
# PROCESS VIDEO
# =========================

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    if BLUR_KERNEL > 1:
        gray = cv2.GaussianBlur(gray, (BLUR_KERNEL, BLUR_KERNEL), 0)

    # Dark object on bright background
    _, mask = cv2.threshold(gray, THRESHOLD_VALUE, 255, cv2.THRESH_BINARY_INV)

    # Clean up the mask a bit
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    ball_contour = choose_ball_contour(contours)

    # Black output frame
    output_frame = np.zeros((height, width, 3), dtype=np.uint8)

    if ball_contour is not None:
        # Draw only the ball in white
        cv2.drawContours(output_frame, [ball_contour], -1, (255, 255, 255), -1)

    out.write(output_frame)

cap.release()
out.release()
cv2.destroyAllWindows()

print("Saved to:", output_path)

Saved to: Z:\Video_Files\Test_ball_only.avi
